# Visualizacao com t-SNE

Neste notebook vamos carregar as matrizes **Bag of Words** e **TF-IDF** geradas no notebook `introducao_pln` e aplicar o **t-SNE** para reduzir a dimensionalidade a duas dimensoes. Assim conseguimos visualizar como as noticias se agrupam segundo cada representacao.

## O que e t-SNE

O **t-SNE** (*t-distributed Stochastic Neighbor Embedding*) e uma tecnica de reducao de dimensionalidade nao linear, muito usada para visualizacao. Ele tenta colocar pontos parecidos no espacao original proximos no espaco reduzido (em geral 2D), preservando a estrutura local dos dados.

Aqui, cada ponto sera uma noticia. Esperamos que noticias com vocabulario parecido apareçam proximas no grafico.

In [2]:
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE

## Carregando os dados

In [3]:
df_bow = pd.read_csv("../dados/bow.csv")
df_tfidf = pd.read_csv("../dados/tfidf.csv")

print(f"BoW:    {df_bow.shape}")
print(f"TF-IDF: {df_tfidf.shape}")
df_bow.head()

BoW:    (1297, 15792)
TF-IDF: (1297, 15791)


,data_publicacao,titulo,categoria,url,imagem,subtitulo,bow_aa,bow_aab,bow_abafa,bow_abaixa,...,bow_zoeira,bow_zona,bow_zonas,bow_zortea,bow_zorz,bow_zubeldia,bow_zudcuxlcuk,bow_zulu,bow_zvezda,palavras_unicas
0,2026-04-14T16:18:41-03:00,Santos acerta renovação de Robinho Jr. até o f...,Santos,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Clube da Baixada Santista define condições de ...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,97
1,2026-04-14T15:52:24-03:00,Cruzeiro anuncia renovação com Artur Jorge apó...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador iniciou no clube celeste em 22 de ma...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,74
2,2026-04-13T18:37:05-03:00,Ex-Corinthians cita caso vivido em clube e cri...,Corinthians,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Betão comentou o episódio recente no Dérbi pau...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,204
3,2026-04-10T09:00:53-03:00,Cruzeiro: Pedro Junio explica escolha por Artu...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,"Ao CNN Esportes S/A deste domingo (12), vice-p...",0,0,0,0,...,0,3,0,0,0,0,0,0,0,139
4,2026-03-12T08:46:42-03:00,"""Eu sou 200% Flamengo"", diz Leonardo Jardim ap...",Flamengo,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador destaca evolução da equipe e comenta...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,116


## Separando metadados e matrizes

As colunas que comecam com `bow_` ou `tfidf_` sao as variaveis numericas. As outras sao metadados (titulo, url, tags, data).

In [4]:
colunas_bow = [c for c in df_bow.columns if c.startswith("bow_")]
colunas_tfidf = [c for c in df_tfidf.columns if c.startswith("tfidf_")]

X_bow = df_bow[colunas_bow].values
X_tfidf = df_tfidf[colunas_tfidf].values

titulos = df_bow["titulo"].tolist()

print(f"X_bow:   {X_bow.shape}")
print(f"X_tfidf: {X_tfidf.shape}")

X_bow:   (1297, 15785)
X_tfidf: (1297, 15785)


## Aplicando o t-SNE

Como temos poucas noticias (16), usamos `perplexity` baixo (o valor precisa ser menor que o numero de amostras).

In [21]:
perplexity = 10
random_state = 42

tsne_bow = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_bow = tsne_bow.fit_transform(X_bow)

tsne_tfidf = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_tfidf = tsne_tfidf.fit_transform(X_tfidf)

print("embed_bow:  ", embed_bow.shape)
print("embed_tfidf:", embed_tfidf.shape)

embed_bow:   (1297, 2)
embed_tfidf: (1297, 2)


## Visualizando lado a lado

In [22]:
def plotar(embed, nome):

    df_plot = pd.DataFrame({
        "x": embed[:, 0],
        "y": embed[:, 1],
        "indice": df_bow.index,
        "titulo": df_bow["titulo"],
        "data": df_bow["data_publicacao"],
    })

    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        hover_data={
            "indice": True,
            "titulo": True,
            "data": True,
            "x": False,
            "y": False
        },
        title=f"t-SNE - {nome}",
        width=700,
        height=700,
    )

    fig.write_html(nome + ".html")

    fig.show()


plotar(embed_bow, "Bag of Words")
plotar(embed_tfidf, "TF-IDF")